In [ ]:
import concurrent.futures
import csv
import os
import sys
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["axes.formatter.useoffset"] = False

root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.models.machines import ieee_machine2_trained_neural_flux
from current_setpoints.models.forward_model import ForwardModel, Fault, count_peaks_at_limit
from current_setpoints.optimization import FeasibilityOptimizer

CURR_MAX, VOLT_MAX = 30.0, 13.0
N_ANGLE_NODES = 90

drive = ieee_machine2_trained_neural_flux(curr_max=CURR_MAX, volt_max=VOLT_MAX)
CONST_MECH_SPEED = 30.0 / (np.pi * drive.n_ppairs)
PHASE_LETTERS = ["a", "b", "c", "d", "e"]

FWD = ForwardModel(drive, fault=Fault(()), n_theta=700)
SOLVER_OPTS = {"disp": False, "ftol": 1e-8, "maxiter": 300}
OPT = FeasibilityOptimizer(FWD, SOLVER_OPTS, n_grid=N_ANGLE_NODES)

OUT_DIR = "../figs/feasibility_sweep_infeasible_healthy_fw1200"

RPM_CENTER = 1200.0
RPM_HALF_SPAN = 100.0

N_OMEGA = 3
N_TORQ_IN = 4
OVERSHOOT_FRACS = (1.10, 1.30)

PROBE_TIMEOUT_SECONDS = 45
POINT_TIMEOUT_SECONDS = 90
N_PROBE_MARGINS = 6

n_probes = 1 + N_OMEGA
worst_probe_time = n_probes * N_PROBE_MARGINS * PROBE_TIMEOUT_SECONDS
n_points = N_OMEGA * (N_TORQ_IN + len(OVERSHOOT_FRACS))
worst_point_time = n_points * POINT_TIMEOUT_SECONDS
worst_case_total = worst_probe_time + worst_point_time
print(f"const_mech_speed = {CONST_MECH_SPEED:.4f}")
print(f"speed band: {RPM_CENTER - RPM_HALF_SPAN:.0f}-{RPM_CENTER + RPM_HALF_SPAN:.0f} RPM, {N_OMEGA} rows")
print(f"{n_probes} T_max probes, {n_points} sweep points")
print(f"worst-case wall-time ceiling: {worst_case_total / 3600:.2f} h "
      f"({worst_probe_time}s probes + {worst_point_time}s points)")
print("This is a hard CEILING (every single solve timing out) -- most points converge, or fail "
      "cheaply at R0, far faster than this, so the actual run should finish well under it.")

In [ ]:
def segment_label(n_curr, n_volt):
    roman = {0: "", 1: "I", 2: "II"}
    return f"MTPA_{n_curr}" if n_volt == 0 else f"FW_{roman[n_volt]},{n_curr}"


def fine_idx_grid(fwd, theta_grid):
    n_theta = fwd.vec_theta.size - 1
    return np.round(theta_grid / (2 * np.pi) * n_theta).astype(int) % n_theta


def reconstruct_phase_currents(fwd, theta_grid, sched):
    idx_grid = fine_idx_grid(fwd, theta_grid)
    n_surv = fwd.phase_map_at_theta(0).shape[0]
    out = np.full((n_surv, len(idx_grid)), np.nan)
    for k, ti in enumerate(idx_grid):
        if np.all(np.isfinite(sched[k])):
            out[:, k] = fwd.phase_map_at_theta(int(ti)) @ sched[k]
    return out


def reconstruct_phase_voltages(fwd, theta_grid, sched, omega):
    idx_grid = fine_idx_grid(fwd, theta_grid)
    kept = list(fwd.fault._kept)
    out = np.full((len(kept), len(idx_grid)), np.nan)
    for k, ti in enumerate(idx_grid):
        i_n = sched[k]
        if np.all(np.isfinite(i_n)):
            gU, _, bV = fwd.volt_map_at_theta(int(ti), omega, i_n)
            out[:, k] = (gU @ i_n + bV)[kept]
    return out


def alpha_beta_xy(theta_grid, sched):
    c1, s1 = np.cos(theta_grid), np.sin(theta_grid)
    c3, s3 = np.cos(3 * theta_grid), np.sin(3 * theta_grid)
    id1, iq1, id3, iq3 = sched[:, 0], sched[:, 1], sched[:, 2], sched[:, 3]
    i_alpha = c1 * id1 - s1 * iq1
    i_beta = s1 * id1 + c1 * iq1
    i_x = c3 * id3 - s3 * iq3
    i_y = s3 * id3 + c3 * iq3
    return i_alpha, i_beta, i_x, i_y


def classify_status(sol):
    """FEASIBLE: converged -- the exact dynamic voltage residual
    (voltage_residuals_exact) is within TOL at every node. R0_INFEASIBLE:
    not even the independent per-node solve (which ignores the inter-node
    coupling term entirely) could hit this torque target everywhere -- a
    harder, purely static failure, expected for the deliberately-overshot
    torque targets in build_torque_grid. STUCK: R0 succeeded everywhere
    but the joint dynamic-voltage repair (the entire reason
    FeasibilityOptimizer exists) could not certify feasibility within
    max_outer_iter -- reported honestly per the class docstring, no
    fallback to any other method."""
    if sol.diagnostics.get("fail_reason") == "R0 failed at some nodes":
        return "R0_INFEASIBLE"
    return "FEASIBLE" if sol.success else "STUCK"


def exact_phi(fwd, opt, sol, omega):
    """Worst-case exact voltage residual (opt.voltage_residuals_exact) --
    the SAME number FeasibilityOptimizer itself used to decide
    feasibility. Only used for the title annotation and the log, not for
    any of the nine plotted panels (those stay identical to
    pointwise_waveform_diagnostics.ipynb/feasibility_sweep.ipynb)."""
    theta_grid = sol.diagnostics["theta_grid"]
    idx_grid = fine_idx_grid(fwd, theta_grid)
    r_exact, _ = opt.voltage_residuals_exact(omega, sol.curr_dq, idx_grid)
    return float(np.nanmax(r_exact)) if np.any(np.isfinite(r_exact)) else float("nan")

In [ ]:
def save_setpoint_plot(fwd, sol, torque, rpm, omega, save_path, status, phi_exact):
    """Identical 9-panel figure to pointwise_waveform_diagnostics.ipynb's
    plot_point / feasibility_sweep.ipynb's save_setpoint_plot. The only
    difference from feasibility_sweep.ipynb: this is called
    unconditionally (that one only calls it when sol.success), so
    STUCK/R0_INFEASIBLE points get the exact same plot too -- whatever
    finite current the solver did produce (R0's own result, or the joint
    solve's last candidate) is still real information about what
    infeasible looks like electrically. A node that never got a finite
    current at all (R0 failed there) just leaves a gap in every curve --
    already how reconstruct_phase_currents/alpha_beta_xy behave. The
    title additionally states the status and the exact residual
    (phi_exact) so infeasible points are identifiable at a glance."""
    sched = sol.curr_dq
    theta_grid = sol.diagnostics["theta_grid"]
    theta = np.rad2deg(theta_grid)

    iph = reconstruct_phase_currents(fwd, theta_grid, sched)
    vph = reconstruct_phase_voltages(fwd, theta_grid, sched, omega)
    i_alpha, i_beta, i_x, i_y = alpha_beta_xy(theta_grid, sched)

    n_curr = count_peaks_at_limit(np.concatenate([iph, iph[:, :1]], axis=1), CURR_MAX)
    n_volt = count_peaks_at_limit(np.concatenate([vph, vph[:, :1]], axis=1), VOLT_MAX) if np.any(np.isfinite(vph)) else 0
    seg_text = segment_label(n_curr, n_volt)

    kept = list(fwd.fault._kept)
    letters = [PHASE_LETTERS[p] for p in kept]

    norm_pw = np.sum(sched ** 2, axis=1)
    cu_pw = np.nansum(iph ** 2, axis=0)
    torque_pw = np.array([
        fwd.drive.torque(omega, sched[k]) if np.all(np.isfinite(sched[k])) else np.nan
        for k in range(sched.shape[0])
    ])

    have_fft = np.all(np.isfinite(iph))
    if have_fft:
        iph_fft = np.abs(np.fft.rfft(iph, axis=1)) / iph.shape[1]
        harmonics = np.arange(iph_fft.shape[1])

    status_color = {"FEASIBLE": "tab:green", "STUCK": "tab:orange", "R0_INFEASIBLE": "tab:red"}[status]
    phi_text = f"phi_exact={phi_exact:+.2f} V" if np.isfinite(phi_exact) else "phi_exact=n/a (R0 gap)"

    fig, ax = plt.subplots(3, 3, figsize=(18, 13))
    fig.suptitle(
        f"[{status}] {seg_text}   T*={torque:.2f} Nm,  {rpm:.1f} RPM   ({phi_text})",
        fontsize=13, color=status_color,
    )

    dq_lbl = [r"$i_{sd}^1$", r"$i_{sq}^1$", r"$i_{sd}^3$", r"$i_{sq}^3$"]
    dq_col = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
    a = ax[0, 0]
    for j in range(fwd.drive.dim):
        a.plot(theta, sched[:, j], color=dq_col[j], label=dq_lbl[j])
    a.set_title("DQ setpoints")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("DQ current [A]")
    a.legend(fontsize=8)

    a = ax[0, 1]
    for p, lt in enumerate(letters):
        a.plot(theta, iph[p], label=f"$i_{{{lt}}}$")
    a.axhline(CURR_MAX, color="gray", ls=":"); a.axhline(-CURR_MAX, color="gray", ls=":")
    a.set_title("Phase currents")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("phase current [A]")
    a.legend(fontsize=8)

    a = ax[0, 2]
    for p, lt in enumerate(letters):
        a.plot(theta, vph[p], label=f"$v_{{{lt}}}$")
    a.axhline(VOLT_MAX, color="gray", ls=":"); a.axhline(-VOLT_MAX, color="gray", ls=":")
    a.set_title("Phase voltages")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("phase voltage [V]")
    a.legend(fontsize=8)

    a = ax[1, 0]
    a.plot(theta, cu_pw, color="tab:orange", label=f"mean={np.nanmean(cu_pw):.1f}")
    a.set_title(r"Copper-loss density $\sum_k i_k^2(\theta)$")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel(r"$\sum_k i_k^2$ [A$^2$]")
    a.legend(fontsize=8)

    a = ax[1, 1]
    a.plot(theta, norm_pw, color="tab:orange", label=r"$\|i_s(\theta)\|^2$")
    a.set_title(r"DQ norm $\|i_s\|^2$")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel(r"$\|i_s\|^2$ [A$^2$]")
    a.legend(fontsize=8)

    a = ax[1, 2]
    a.plot(i_alpha, i_beta, color="tab:purple")
    a.set_title(r"$\alpha\beta$ current")
    a.set_xlabel(r"$i_\alpha$ [A]"); a.set_ylabel(r"$i_\beta$ [A]")
    a.set_aspect("equal")

    a = ax[2, 0]
    a.plot(i_x, i_y, color="tab:brown")
    a.set_title("xy current")
    a.set_xlabel(r"$i_x$ [A]"); a.set_ylabel(r"$i_y$ [A]")
    a.set_aspect("equal")

    a = ax[2, 1]
    a.plot(theta, torque_pw, color="tab:red", label=f"mean={np.nanmean(torque_pw):.3f}")
    a.axhline(torque, color="gray", ls=":", label=f"T*={torque:.2f}")
    a.set_title("Achieved torque (per-node constraint check)")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("torque [Nm]")
    a.legend(fontsize=8)

    a = ax[2, 2]
    if have_fft:
        width = 0.8 / len(letters)
        for p, lt in enumerate(letters):
            a.bar(harmonics + p * width, iph_fft[p], width=width, label=f"$i_{{{lt}}}$")
        a.set_xlim(-0.5, min(15, harmonics[-1]) + 0.5)
        a.set_title("Phase current spectrum (electrical harmonics)")
        a.set_xlabel("harmonic order"); a.set_ylabel("amplitude [A]")
        a.legend(fontsize=8)
    else:
        a.axis("off")
        a.text(0.5, 0.5, "spectrum skipped:\nsome nodes infeasible/NaN",
               ha="center", va="center", transform=a.transAxes)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(save_path, dpi=110)
    plt.close(fig)

In [ ]:
def call_with_timeout(timeout, fn, *args, **kwargs):
    """Run fn(*args, **kwargs) with a hard wall-clock cap. If it doesn't
    finish in time, returns (None, "timeout") and moves on -- the
    underlying SLSQP call keeps running in its own thread in the
    background (Windows has no reliable way to kill a specific blocking
    C-extension call from another thread), but the sweep loop itself is
    never blocked past `timeout` seconds by any single solve. This is
    the actual mechanism behind "it shouldn't start hanging anywhere":
    every solve below goes through this, so the loop always moves on
    regardless of what SLSQP does in the background.
    """
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    future = executor.submit(fn, *args, **kwargs)
    try:
        result = future.result(timeout=timeout)
        executor.shutdown(wait=False)
        return result, None
    except concurrent.futures.TimeoutError:
        executor.shutdown(wait=False)
        return None, "timeout"
    except Exception as exc:
        executor.shutdown(wait=False)
        return None, f"exception: {exc!r}"


def probe_t_max(opt, omega, guess=None):
    """Robust T_max probe (same mechanism as feasibility_sweep.ipynb):
    opt.maximize_torque() picks t_max as the min, across all angle nodes,
    of each node's OWN individually-maximal torque, then asks
    minimize_current() to hit that value EXACTLY -- precisely the tangent
    point of that node's own feasible boundary, a notoriously hard spot
    for SLSQP to converge to exactly. Back the target off by a small
    margin, retrying with a larger backoff if it still doesn't converge.

    Returns (t_max, curr_dq, status):
      "ok"          -- converged; t_max/curr_dq are usable as this row's ceiling
      "physical"    -- R0's own per-node torque ceiling is <= 0: the machine
                       genuinely cannot produce positive torque here at all
      "no_converge" -- ceiling IS positive but no backoff margin converged;
                       NOT evidence of infeasibility, just a probe failure

    Each backoff attempt is capped at PROBE_TIMEOUT_SECONDS, not the
    larger POINT_TIMEOUT_SECONDS -- this loop already retries up to
    N_PROBE_MARGINS margins on its own.
    """
    maps = opt._precompute_maps(omega)
    r0_grid, ok0 = opt._run_per_angle(omega, maps, None, maximize=True)
    if not np.any(ok0):
        return None, None, "no_converge"
    torques = np.array([opt.fwd.drive.torque(omega, r0_grid[i]) if ok0[i] else np.nan for i in range(opt.n_grid)])
    t_max_raw = float(np.nanmin(torques))

    if t_max_raw <= 0.0:
        return None, None, "physical"

    for margin in (1.0, 0.999, 0.995, 0.99, 0.95, 0.9):
        sol, err = call_with_timeout(PROBE_TIMEOUT_SECONDS, opt.minimize_current, t_max_raw * margin, omega, guess=guess)
        if sol is not None and sol.success:
            return sol.torque, sol.curr_dq, "ok"
    return None, None, "no_converge"

In [ ]:
LOG_FIELDS = ["torque", "rpm", "status", "phi_exact", "n_active", "seconds", "reason"]


def append_log_row(log_path, row):
    """Write one row immediately, not buffered until the end -- a run
    interrupted mid-sweep still leaves every point up to that moment
    logged AND plotted (PNGs are written incrementally too), same
    crash-resilience as feasibility_sweep.ipynb."""
    is_new = not os.path.exists(log_path)
    with open(log_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if is_new:
            writer.writeheader()
        writer.writerow(row)


def build_torque_grid(ceiling):
    """In-range points spanning [0, ceiling], PLUS len(OVERSHOOT_FRACS)
    points deliberately ABOVE it. `ceiling` is either this row's own R0
    torque ceiling (probe status "ok") or, when that couldn't be
    established (status "physical"/"no_converge"), the global one --
    either way the overshoot points then sit above whatever this row can
    actually do, so every row contributes at least len(OVERSHOOT_FRACS)
    guaranteed-infeasible (R0_INFEASIBLE) points, without relying on
    chance near the true feasibility boundary."""
    in_range = np.linspace(0.0, ceiling, N_TORQ_IN)
    over = np.array([ceiling * f for f in OVERSHOOT_FRACS])
    return np.concatenate([in_range, over])


def run_sweep():
    os.makedirs(OUT_DIR, exist_ok=True)
    log_path = os.path.join(OUT_DIR, "sweep_log.csv")

    omega_ref = (RPM_CENTER - RPM_HALF_SPAN) / CONST_MECH_SPEED
    torq_max_global, _, status_global = probe_t_max(OPT, omega_ref)
    if torq_max_global is None:
        raise RuntimeError(f"reference T_max probe at {RPM_CENTER - RPM_HALF_SPAN:.0f} RPM "
                            f"returned '{status_global}' -- cannot size the sweep.")
    print(f"reference T_max ~ {torq_max_global:.3f} Nm at {RPM_CENTER - RPM_HALF_SPAN:.0f} RPM")

    vec_omega = np.linspace(RPM_CENTER - RPM_HALF_SPAN, RPM_CENTER + RPM_HALF_SPAN, N_OMEGA) / CONST_MECH_SPEED
    log_rows = []
    prev_max_curr = None
    prev_curr = None
    t_start = time.perf_counter()

    for idx_omega, omega in enumerate(vec_omega):
        omega = float(omega)
        rpm = omega * CONST_MECH_SPEED

        torq_max_local, curr_at_max, status = probe_t_max(OPT, omega, guess=prev_max_curr)
        if status == "ok":
            ceiling = torq_max_local
            prev_max_curr = curr_at_max
            print(f"speed {idx_omega + 1}/{N_OMEGA}: {rpm:.1f} RPM  (row ceiling ~ {ceiling:.3f} Nm)")
        else:
            ceiling = torq_max_global
            print(f"speed {idx_omega + 1}/{N_OMEGA}: {rpm:.1f} RPM  "
                  f"-- row T_max probe returned '{status}', using reference ceiling ~ {ceiling:.3f} Nm instead "
                  f"(every point this row is then expected to be at least STUCK or R0_INFEASIBLE)")

        vec_torq = build_torque_grid(ceiling)

        for torque in vec_torq:
            torque = float(torque)
            fname = f"T{torque:07.3f}_RPM{rpm:07.1f}.png".replace(" ", "0")
            fpath = os.path.join(OUT_DIR, fname)

            if os.path.exists(fpath):
                print(f"    T={torque:.3f} Nm  already done, skipping")
                continue

            t0 = time.perf_counter()
            sol, err = call_with_timeout(POINT_TIMEOUT_SECONDS, OPT.minimize_current, torque, omega, guess=prev_curr)
            dt = time.perf_counter() - t0

            if sol is None:
                row = {"torque": torque, "rpm": rpm, "status": "TIMEOUT_OR_ERROR", "phi_exact": "",
                       "n_active": "", "seconds": dt, "reason": err}
                print(f"    T={torque:.3f} Nm  {err}  ({dt:.1f}s)")
                log_rows.append(row); append_log_row(log_path, row)
                continue

            status_pt = classify_status(sol)
            if status_pt == "FEASIBLE":
                prev_curr = sol.curr_dq

            phi_exact = exact_phi(FWD, OPT, sol, omega)
            n_active = len(sol.diagnostics.get("active", []))

            try:
                save_setpoint_plot(FWD, sol, torque, rpm, omega, fpath, status_pt, phi_exact)
                plot_ok, plot_err = True, ""
            except Exception as exc:
                plot_ok, plot_err = False, f"plot exception: {exc!r}"

            row_status = status_pt if plot_ok else f"{status_pt}(PLOT_FAILED)"
            row = {"torque": torque, "rpm": rpm, "status": row_status, "phi_exact": phi_exact,
                   "n_active": n_active, "seconds": dt, "reason": plot_err}
            print(f"    T={torque:.3f} Nm  {status_pt}  phi_exact={phi_exact:+.2f}V  ({dt:.1f}s)"
                  + ("" if plot_ok else f"  PLOT FAILED: {plot_err}"))
            log_rows.append(row); append_log_row(log_path, row)

    elapsed = time.perf_counter() - t_start
    n_ok = sum(1 for r in log_rows if r["status"] == "FEASIBLE")
    print(f"\ndone in {elapsed / 3600:.2f} h: {n_ok}/{len(log_rows)} feasible, log at {log_path}")
    return log_rows

In [ ]:
log_rows = run_sweep()

In [ ]:
def read_sweep_log():
    """Read the log back off disk rather than trusting log_rows in memory
    -- lets this cell be re-run to summarize a sweep that is still
    running (rows are flushed one at a time) or one resumed after being
    interrupted. Later rows win on a duplicate (rpm, torque) key, which is
    what a resumed run produces."""
    log_path = os.path.join(OUT_DIR, "sweep_log.csv")
    if not os.path.exists(log_path):
        return {}
    points = {}
    with open(log_path, newline="") as f:
        for row in csv.DictReader(f):
            key = (round(float(row["rpm"]), 3), round(float(row["torque"]), 6))
            points[key] = row
    return points


def plot_feasibility_map(save_path=None):
    """Torque/speed scatter, colored by exactly the three outcomes this
    sweep can log (plus a 4th, gray, bucket for a solve that timed out or
    raised before it could even be classified) -- every attempted point
    is shown, since nothing here was skipped the way the ceiling-gated
    "above T_max" points were in feasibility_sweep.ipynb."""
    save_path = save_path if save_path is not None else os.path.join(OUT_DIR, "feasibility_map.png")
    pts = read_sweep_log()
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = {"FEASIBLE": "tab:green", "STUCK": "tab:orange", "R0_INFEASIBLE": "tab:red"}
    markers = {"FEASIBLE": "o", "STUCK": "^", "R0_INFEASIBLE": "x"}

    known = set()
    for stat, color in colors.items():
        xy = [(float(p["rpm"]), float(p["torque"])) for p in pts.values() if p["status"] == stat]
        known.update(k for k, p in pts.items() if p["status"] == stat)
        if xy:
            ax.scatter(*zip(*xy), c=color, marker=markers[stat], s=90,
                       edgecolors="black", linewidths=0.4, label=f"{stat} ({len(xy)})", zorder=3)

    other = [(float(p["rpm"]), float(p["torque"])) for k, p in pts.items() if k not in known]
    if other:
        ax.scatter(*zip(*other), c="gray", marker="s", s=70, label=f"timeout/error ({len(other)})", zorder=2)

    n_feasible = sum(1 for p in pts.values() if p["status"] == "FEASIBLE")
    ax.set_title(f"healthy mode: {n_feasible}/{len(pts)} feasible")
    ax.set_xlabel("speed [RPM]"); ax.set_ylabel("torque [Nm]")
    ax.grid(alpha=0.3, zorder=0)
    ax.legend(fontsize=9)
    plt.tight_layout()
    fig.savefig(save_path, dpi=110)
    plt.close(fig)
    print(f"saved {save_path}")


plot_feasibility_map()
for stat in ("FEASIBLE", "STUCK", "R0_INFEASIBLE"):
    n = sum(1 for r in log_rows if r["status"] == stat)
    print(f"  {stat:14s} {n}")